In [1]:
!pip install stanza
import stanza
stanza.download('en')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.7/773.7 kB 95.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.7/418.7 kB 56.1 MB/s eta 0:00:00


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Downloading default packages for language: en (English) ...


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/en/default.zip
INFO:stanza:Finished downloading models and saved to /root/.cache/stanza/1.11.0/resources


[['zip', 'default.zip']]

In [2]:
import pandas as pd

sentence_level_df = pd.read_csv("alias_sentence_features_primary_sorted_with_type.csv")

In [3]:
import pandas as pd

def is_dialogue(sentence):
    if isinstance(sentence, str):
        return int(any(q in sentence for q in ['"', '“', '”', '‘', '’']))
    return 0

# Apply to your DataFrame
sentence_level_df['is_dialogue'] = sentence_level_df['text'].apply(is_dialogue)
sentence_level_df

,story_id,person,type,aliases,sentence_id,text,mention_count,word_count,text_prev,text_next,bert_context,is_primary_in_sentence,is_dialogue
0,1,Person-1,protagonist,['Man'],19,7 The Book that begins with a Murder Mystery c...,5,110.0,After you have seen the Pictures there is no n...,You know at once that the most Respected and l...,After you have seen the Pictures there is no n...,1,0
1,1,Person-1,protagonist,['Man'],21,8 The Book that gets away with one Man asking ...,5,110.0,You know at once that the most Respected and l...,The Man who asks this Question has a Name whic...,You know at once that the most Respected and l...,1,0
2,1,Person-1,protagonist,['Man'],22,The Man who asks this Question has a Name whic...,5,110.0,8 The Book that gets away with one Man asking ...,You feel instinctively that he is going to be ...,8 The Book that gets away with one Man asking ...,1,0
3,1,Person-1,protagonist,['Man'],24,is reached but who can take any real Interest ...,5,110.0,You feel instinctively that he is going to be ...,9 The Book that tells all about Society and ho...,You feel instinctively that he is going to be ...,1,0
4,1,Person-1,protagonist,['Man'],27,The clever Man of the World who says all the K...,5,110.0,Even the Women drink Brandy and Soda smoke Cig...,An irritable Person after reading nine Chapter...,Even the Women drink Brandy and Soda smoke Cig...,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4615,198,Person-5,neutral,['rabbit'],6,And he can out-run a jack rabbit when he gets ...,1,12.0,That's because he's traveled on the side-hills...,Dangerous,That's because he's traveled on the side-hills...,0,0
4616,198,Person-6,neutral,['wife'],87,O my wife,1,3.0,My God,it says,My God [SEP] O my wife [SEP] it says,1,0
4617,199,Person-1,protagonist,['giant'],6,Many times I walked through that valley and lo...,2,82.0,And once I walked through a golden valley that...,And alway the goal of my fancies was the might...,And once I walked through a golden valley that...,1,0
4618,199,Person-1,protagonist,['giant'],19,From beyond came a glow that weirdly lit the g...,2,82.0,Last night I swallowed the drug and floated dr...,But as the gate swung wider and the sorcery of...,Last night I swallowed the drug and floated dr...,1,0


In [5]:
# Load pipeline
import stanza

# Load the Indonesian model once
nlp = stanza.Pipeline(lang='en', processors='tokenize,pos,lemma')

# Fungsi untuk membaca file lexicon
def load_afinn(afinn_path):
    lexicon = {}
    for path in [afinn_path]:  # Tidak perlu pakai multiplier lagi
        with open(path, encoding='utf-8') as f:
            lines = f.readlines()[1:]  # skip header
            for line in lines:
                if line.strip():
                    parts = line.strip().split()
                    if len(parts) >= 2:
                        word = parts[0]
                        try:
                            score = float(parts[1])
                            lexicon[word] =  score
                        except ValueError:
                            continue  # skip if score is not a float
    return lexicon

def verb_adj_sentiment_stanza(text, lexicon):
    if pd.isna(text):
        return 0

    text = str(text).strip()
    if text == "":
        return 0

    doc = nlp(text)

    count = 0
    for sent in doc.sentences:
        for word in sent.words:
            if word.upos in ['VERB', 'ADJ'] and word.lemma.lower() in lexicon:
                count += 1
    return count

afinn_path = "AFINN.tsv"
lexicon = load_afinn(afinn_path)

# Terapkan ke semua baris dan kembalikan dua kolom baru
sentence_level_df['count_lexicon'] = sentence_level_df['text'].apply(
    lambda x: verb_adj_sentiment_stanza(x, lexicon)
)

INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Loading these models for language: en (English):
| Processor | Package           |
---------------------------------
| tokenize  | combined          |
| mwt       | combined          |
| pos       | combined_charlm   |
| lemma     | combined_nocharlm |

INFO:stanza:Using device: cuda
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: mwt
INFO:stanza:Loading: pos
INFO:stanza:Loading: lemma
INFO:stanza:Done loading processors!


In [6]:
import stanza
import pandas as pd
import ast

# Load English NLP pipeline
nlp = stanza.Pipeline('en', processors='tokenize,pos,lemma,depparse', use_gpu=False)

def safe_parse_aliases(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return []
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return parsed
            return [x]
        except:
            return [x]
    return []

def get_alias_roles(row):
    sentence = row['text']
    aliases = safe_parse_aliases(row['aliases'])

    # amankan sentence
    if pd.isna(sentence):
        return pd.Series({'is_subject': 0, 'is_object': 0})

    sentence = str(sentence).strip()
    if not sentence:
        return pd.Series({'is_subject': 0, 'is_object': 0})

    is_subject = 0
    is_object = 0

    doc = nlp(sentence)

    for sent in doc.sentences:
        for word in sent.words:
            for alias in aliases:
                alias = str(alias).strip().lower()
                if not alias:
                    continue

                # exact match
                if alias == word.text.lower():
                    if word.deprel in ["nsubj", "nsubj:pass"]:
                        is_subject = 1
                    elif word.deprel in ["obj", "iobj"]:
                        is_object = 1

    return pd.Series({'is_subject': is_subject, 'is_object': is_object})

sentence_level_df[['is_subject', 'is_object']] = sentence_level_df.apply(get_alias_roles, axis=1)

INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Loading these models for language: en (English):
| Processor | Package           |
---------------------------------
| tokenize  | combined          |
| mwt       | combined          |
| pos       | combined_charlm   |
| lemma     | combined_nocharlm |
| depparse  | combined_charlm   |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: mwt
INFO:stanza:Loading: pos
INFO:stanza:Loading: lemma
INFO:stanza:Loading: depparse
INFO:stanza:Done loading processors!


In [7]:
print(sentence_level_df.columns)

Index(['story_id', 'person', 'type', 'aliases', 'sentence_id', 'text',
       'mention_count', 'word_count', 'text_prev', 'text_next', 'bert_context',
       'is_primary_in_sentence', 'is_dialogue', 'count_lexicon', 'is_subject',
       'is_object'],
      dtype='object')


In [9]:
sentence_level_df.to_csv("data_inggris_ferro_base.csv", index=False)

In [10]:
import pandas as pd

# Step 1: Identify the least frequent label
label_counts = sentence_level_df['type'].value_counts()
least_common_label = label_counts.idxmin()

# Step 2: Sort so that the least common label comes first (we want to keep these)
sentence_level_df_sorted = sentence_level_df.sort_values(
    by='type',
    key=lambda x: x.apply(lambda val: 0 if val == least_common_label else 1)
)

# Step 3: Drop duplicates in 'subject_sentences' keeping the first occurrence (which is from least_common_label)
sentence_level_df = sentence_level_df_sorted.drop_duplicates(subset='text', keep='first')


In [13]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 138.4 MB/s eta 0:00:00


In [17]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [18]:
import re
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from gensim.models import Word2Vec
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Stopwords bahasa Inggris
ENGLISH_STOPWORDS = set(stopwords.words('english'))

# Stemmer bahasa Inggris
stemmer = PorterStemmer()

def preprocess_sentence(sentence):
    # Handle missing value
    if pd.isna(sentence):
        return ""

    # Lowercase
    sentence = sentence.lower()

    # Tokenize: ambil kata alphabet saja
    tokens = re.findall(r'\b[a-zA-Z]+\b', sentence)

    # Remove stopwords
    tokens = [token for token in tokens if token not in ENGLISH_STOPWORDS]

    # Stem each token
    stemmed_tokens = [stemmer.stem(token) for token in tokens]

    return " ".join(stemmed_tokens)

sentence_level_df["subject_sentences_stem"] = sentence_level_df["text"].apply(preprocess_sentence)

/tmp/ipykernel_2407/2364525096.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sentence_level_df["subject_sentences_stem"] = sentence_level_df["text"].apply(preprocess_sentence)


In [20]:
from sklearn.model_selection import train_test_split
# Split train-test
train_df, test_df = train_test_split(
    sentence_level_df,
    test_size=0.2,
    stratify=sentence_level_df['type'],
    random_state=42
)

# Pilih fitur dan label
X_train_exploded = train_df.copy()
y_train_exploded = train_df['type']

X_test_exploded = test_df.copy()
y_test_exploded = test_df['type']


In [21]:
train_df['type'].value_counts()

,count
type,
neutral,1708
protagonist,1112
antagonist,20


In [22]:
import pandas as pd
from collections import Counter
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
# Function to get the most common element from a list
def get_most_common_label(label_list):
    # Flatten the list if it contains nested lists
    # flat_list = [item for sublist in label_list for item in sublist]
    most_common = Counter(label_list).most_common(1)[0][0]
    return most_common

def evaluation_metric_val(best_model, val_df):
    data_test = val_df.copy()
    predictions = best_model.predict(data_test)
    data_test['predicted_type'] = predictions

    # Join alias lists into a string
    data_test["aliases"] = data_test["aliases"].apply(lambda x: ", ".join(x) if isinstance(x, list) else x)

    # Group by entity and aggregate predictions and true labels
    data_test = data_test.groupby(["aliases", "story_id", "person"])[["predicted_type", "type"]].agg(list).reset_index()

    # Get most common label per group
    data_test['predicted_type_group'] = data_test['predicted_type'].apply(get_most_common_label)
    data_test['type_grouped'] = data_test['type'].apply(get_most_common_label)

    # Ground truth and predictions
    y_true = data_test['type_grouped']
    y_pred = data_test['predicted_type_group']

    # Compute confusion matrix
    cm = confusion_matrix(y_true, y_pred)

    # Display confusion matrix
    print("Confusion Matrix:\n", cm)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap='Blues')

    # Print classification report
    print("Classification Report:")
    print(classification_report(y_true, y_pred, digits=4))

    # Compute and print individual metrics
    precision = precision_score(y_true, y_pred, average='macro')
    recall = recall_score(y_true, y_pred, average='macro')
    f1 = f1_score(y_true, y_pred, average='macro')
    accuracy = accuracy_score(y_true, y_pred)

    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-score:  {f1:.4f}")
    print(f"Accuracy:  {accuracy:.4f}")
    return data_test


In [23]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

import os
import joblib
from google.colab import drive

drive.mount('/content/drive')

save_dir = "/content/drive/MyDrive/ML_CharacterType/ferro_svm"
os.makedirs(save_dir, exist_ok=True)

# Preprocessing pipeline
# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('integer', MinMaxScaler(), ["is_dialogue", "is_subject", "count_lexicon"]),
        ('text', TfidfVectorizer(), 'subject_sentences_stem')
    ]
)
# Pipeline with LinearSVC
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LinearSVC())
])

# Grid search parameters
param_grid = {
    'preprocessor__text__max_df': [0.75, 1.0],
    'preprocessor__text__ngram_range': [(1, 1), (1, 2)],
    'preprocessor__text__max_features': [100, 500, 1000, 1500, 2000, 2500, 3000, 5000],
    'classifier__C': [0.1, 1, 10],  # Regularization parameter for SVM
    'classifier__loss': ['hinge', 'squared_hinge'],  # SVM loss functions
    'classifier__max_iter': [1000, 5000],  # max iterations
    'classifier__class_weight': [None, 'balanced']
}

# GridSearchCV
grid_search = GridSearchCV(pipeline, param_grid, cv=5, n_jobs=-1, verbose=1)

# Fit grid search
grid_search.fit(X_train_exploded, y_train_exploded)

# Best parameters and score
print("Best parameters:", grid_search.best_params_)
print("Best cross-validation score:", grid_search.best_score_)

# Evaluate on test set
test_score = grid_search.score(X_test_exploded, y_test_exploded)
y_pred = grid_search.best_estimator_.predict(X_test_exploded)

print("Test set score:", test_score)
report = classification_report(y_test_exploded, y_pred)
print("Classification Report for Best Model:\n", report)

best_model_one_svm_exp_tf = grid_search.best_estimator_
train_accuracy = best_model_one_svm_exp_tf.score(X_train_exploded, y_train_exploded)
test_accuracy = best_model_one_svm_exp_tf.score(X_test_exploded, y_test_exploded)

print("Train Accuracy:", train_accuracy)
print("Test Accuracy:", test_accuracy)

joblib.dump(best_model_one_svm_exp_tf, os.path.join(save_dir, "best_model.pkl"))

with open(os.path.join(save_dir, "best_params.txt"), "w") as f:
    f.write("Best parameters:\n")
    f.write(str(grid_search.best_params_))
    f.write("\n\nBest cross-validation score:\n")
    f.write(str(grid_search.best_score_))
    f.write("\n\nTrain Accuracy:\n")
    f.write(str(train_accuracy))
    f.write("\n\nTest Accuracy:\n")
    f.write(str(test_accuracy))

with open(os.path.join(save_dir, "classification_report.txt"), "w") as f:
    f.write(report)

pred_df = X_test_exploded.copy()
pred_df["y_true"] = y_test_exploded.values if hasattr(y_test_exploded, "values") else y_test_exploded
pred_df["y_pred"] = y_pred
pred_df.to_csv(os.path.join(save_dir, "test_predictions.csv"), index=False)

print(f"Semua hasil berhasil disimpan di: {save_dir}")


Mounted at /content/drive
Fitting 5 folds for each of 768 candidates, totalling 3840 fits
Best parameters: {'classifier__C': 1, 'classifier__class_weight': 'balanced', 'classifier__loss': 'hinge', 'classifier__max_iter': 1000, 'preprocessor__text__max_df': 0.75, 'preprocessor__text__max_features': 3000, 'preprocessor__text__ngram_range': (1, 2)}
Best cross-validation score: 0.7183098591549296
Test set score: 0.7130801687763713
Classification Report for Best Model:
               precision    recall  f1-score   support

  antagonist       1.00      0.80      0.89         5
     neutral       0.75      0.79      0.77       428
 protagonist       0.65      0.59      0.62       278

    accuracy                           0.71       711
   macro avg       0.80      0.73      0.76       711
weighted avg       0.71      0.71      0.71       711

Train Accuracy: 0.901056338028169
Test Accuracy: 0.7130801687763713
Semua hasil berhasil disimpan di: /content/drive/MyDrive/ML_CharacterType/ferro_s